# 06 · Retryable ingestion and provenance

## The engineering question
What happens when Qdrant accepts a batch but the ingestion process crashes before recording completion?

**Learning outcomes:** trace a retryable state transition; use stable tenant-scoped point IDs; retain source revision/hash/span provenance; reconcile remote points with a local manifest; prevent deleted revisions from reappearing.

SQLite and Qdrant do not share a transaction. We first record a pending event containing the desired source revision. We then write deterministic points, remove obsolete IDs, and commit the manifest. On retry, the same IDs are written again. This provides convergence under the tested single-writer failure boundary; it is not a distributed exactly-once guarantee.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("06", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from uuid import uuid4
from vector_masterclass.store import VectorIndex
from vector_masterclass.ingestion import Ingestor
from vector_masterclass.data import documents

index = VectorIndex("vmc_06_" + uuid4().hex[:8])
index.create()
pipeline = Ingestor(index, lesson.path / "manifest.sqlite", words=24, overlap=4)
selected = documents(tenant="public")[:3]
rows = []
try:
    interrupted = False
    try:
        pipeline.ingest(selected, fail_after=1)
    except InterruptedError as error:
        interrupted = True
        print("Expected injected boundary failure:", error)
    rows.append(
        {
            "stage": "interrupted",
            "remote_points": len(index.all_points()),
            "pending_events": len(pipeline.pending()),
            "consistent": pipeline.reconcile()["consistent"],
        }
    )
    pipeline.replay()
    rows.append(
        {
            "stage": "replayed",
            "remote_points": len(index.all_points()),
            "pending_events": len(pipeline.pending()),
            "consistent": pipeline.reconcile()["consistent"],
        }
    )
    repeat = pipeline.ingest(selected)
    rows.append(
        {
            "stage": "same sources again",
            "remote_points": len(index.all_points()),
            "pending_events": len(pipeline.pending()),
            "consistent": pipeline.reconcile()["consistent"],
        }
    )
    revised = selected[0].model_copy(
        update={
            "revision": "2",
            "text": selected[0].text + "\n\nRevised service contact: local support desk.",
        }
    )
    pipeline.ingest([revised])
    revised_points = [p for p in index.all_points() if p.payload["document_id"] == revised.id]
    all_current = all(p.payload["document_key"] == revised.key for p in revised_points)
    pipeline.delete(revised.id, revised.tenant)
    deletion_reconciled = pipeline.reconcile()["consistent"]
    resurrection_rejected = False
    try:
        pipeline.ingest([revised])
    except ValueError:
        resurrection_rejected = True
    events = pd.DataFrame(pipeline.events())
    display(events[["sequence", "source_id", "operation", "state"]])
    lesson.table(events.drop(columns=["source_json"]), "events")
finally:
    pipeline.close()
    index.drop()
    index.close()
results = lesson.table(pd.DataFrame(rows))
display(results)
fig, ax = plt.subplots()
ax.bar(results.stage, results.pending_events, color="#f59e0b")
ax.set(ylabel="Pending events", title="Replay repairs the remote-write/local-commit gap")
lesson.figure(fig)
plt.show()

## Failure, exercise, and production implications
The injected failure occurs after a remote upsert, so a local rollback alone cannot undo it. Replaying the pending desired state converges. The implementation refuses to replay events under a different chunking/model pipeline and refuses changed text under an unchanged source revision.

**Exercise:** set the fault boundary to zero, then replay. Inspect stable IDs and source spans before and after. Why must tenant identity be part of the ID even if document text and revision are identical? What worker-leasing or concurrency control would be needed before adding several ingestion workers?

This lesson uses one writer and explicit deletion events. Missing a document from an input batch does not implicitly delete it. That avoids accidentally treating a partial crawl as an authoritative full snapshot. For production, define snapshot boundaries, tombstones, ordering, reconciliation, and recovery ownership explicitly.

In [ ]:
manifest = lesson.finish(
    {
        "failure_injected": interrupted,
        "replay_converged": bool(results.iloc[1]["consistent"]),
        "rerun_idempotent": repeat["enqueued"] == 0,
        "old_revision_removed": all_current,
        "deletion_reconciled": deletion_reconciled,
        "deleted_revision_not_resurrected": resurrection_rejected,
    }
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.